# NB03 – Data Analysis

## Research Question

**How do brewery characteristics vary across the United States?**

## Purpose

NB01 collected brewery data from the Open Brewery DB API, and NB02 transformed the raw API responses into analysis-ready datasets. The purpose of this notebook is to answer the research question by analysing brewery types and geographic patterns across the United States.

Rather than presenting a series of unrelated visualisations, the analysis is organised around three central findings. Each finding is supported with evidence from the processed data and contributes to answering the research question.

In [1]:
%pip install plotly


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Import Libraries

The analysis uses pandas to manipulate the processed datasets and Plotly Express to create interactive visualisations. Loading the cleaned datasets prepared in NB02 ensures that every result presented in this notebook is based on a consistent and reproducible workflow.

In [3]:
import pandas as pd
from pathlib import Path
import plotly.express as px

## Load the Processed Data

The processed datasets created in NB02 are loaded before any analysis begins. Using the prepared data keeps the workflow reproducible and avoids repeating data cleaning steps.

In [4]:
processed_folder = Path("../data/processed")

breweries_df = pd.read_csv(processed_folder / "breweries_clean.csv")
map_df = pd.read_csv(processed_folder / "breweries_map.csv")

In [5]:
print(f"Complete dataset: {len(breweries_df)} breweries")
print(f"Mapping dataset: {len(map_df)} breweries")

Complete dataset: 2000 breweries
Mapping dataset: 1622 breweries


In [6]:
breweries_df.head()

,id,name,brewery_type,address_1,city,state_province,postal_code,country,longitude,latitude,phone,website_url,state,street
0,ae7b3174-8be8-4d53-a3a5-9b8240970eea,'s,brewpub,1 Friesener Straße,Kronach,Bayern,96317,Germany,11.327765,50.241246,+49 9261 628000,http://www.antla.de,Bayern,1 Friesener Straße
1,aa7cbe9b-3a0f-4888-9884-6186b0042b55,’t Drankorgel,micro,31a Geelsebaan,Mol,Vlaanderen,2400,Belgium,5.056955,51.176258,+32 496 76 40 63,http://tdrankorgel.be/,Vlaanderen,31a Geelsebaan
2,1034754f-abf9-4b42-bd6d-37e110ba4b1a,'T Kroontje,micro,62 Hogebrug,Lebbeke,Vlaanderen,9280,Belgium,4.091393,51.007888,+32 495 43 33 25,http://www.tkroontje.be/,Vlaanderen,62 Hogebrug
3,5128df48-79fc-4f0f-8b52-d06be54d0cec,(405) Brewing Co,micro,1716 Topeka St,Norman,Oklahoma,73069-8224,United States,-97.468182,35.257389,4058160490,http://www.405brewing.com,Oklahoma,1716 Topeka St
4,9c5a66c8-cc13-416f-a5d9-0a769c87d318,(512) Brewing Co,micro,407 Radam Ln Ste F200,Austin,Texas,78745-1197,United States,NaN,NaN,5129211545,http://www.512brewing.com,Texas,407 Radam Ln Ste F200


### Interpretation

The processed datasets loaded successfully and are ready for analysis. The complete dataset contains all 2,000 breweries collected from the API, while the mapping dataset contains the 1,622 breweries with valid geographic coordinates. Using two prepared datasets allows each analysis to use the most appropriate data without repeating preparation steps from NB02.

## Finding 1: Brewery Types Are Concentrated in a Small Number of Categories

A useful starting point is to examine the distribution of brewery types. If only a small number of categories account for most breweries, this suggests that the U.S. brewing industry is dominated by particular business models rather than being evenly distributed across all brewery types.

The following analysis calculates the number of breweries in each category and visualises the results using an interactive bar chart.

In [7]:
brewery_counts = (
    breweries_df["brewery_type"]
    .value_counts()
    .reset_index()
)

brewery_counts.columns = ["brewery_type", "count"]

brewery_counts

,brewery_type,count
0,micro,976
1,brewpub,656
2,closed,127
3,planning,91
4,large,41
5,contract,38
6,regional,35
7,proprietor,11
8,taproom,10
9,bar,9


In [8]:
%pip install nbformat


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
fig = px.bar(
    brewery_counts,
    x="brewery_type",
    y="count",
    title="Number of Breweries by Brewery Type",
    labels={
        "brewery_type": "Brewery Type",
        "count": "Number of Breweries"
    },
    text="count"
)

fig.update_layout(
    xaxis_tickangle=-30
)

fig.update_traces(textposition="outside")

fig.show()


### Interpretation

The distribution of brewery types is highly concentrated rather than evenly spread across categories. Micro breweries are the most common type in the dataset, followed by brewpubs, while every other category contains substantially fewer breweries.

This suggests that the U.S. brewing industry is dominated by smaller-scale brewing operations rather than a balanced mix of business models. Understanding this distribution provides important context for the remaining analyses because brewery type is one of the defining characteristics of the industry.

## Finding 2: Breweries Are Concentrated in a Small Number of States

The research question also asks how brewery characteristics vary across the United States. Examining the number of breweries in each state shows whether breweries are evenly distributed across the country or concentrated in particular regions.

In [10]:
state_counts = (
    breweries_df["state"]
    .value_counts()
    .reset_index()
)

state_counts.columns = ["state", "count"]

state_counts.head(10)

,state,count
0,California,159
1,Bayern,93
2,Washington,85
3,Colorado,70
4,Texas,69
5,Michigan,69
6,New York,60
7,Florida,55
8,Oregon,54
9,Virginia,53


### Preparing the State Analysis

Although the dataset contains breweries from multiple countries, the research question focuses specifically on the United States. Therefore, breweries located outside the United States are excluded from this analysis so that state-level comparisons accurately reflect the U.S. brewing industry.

In [11]:
us_breweries = breweries_df[
    breweries_df["country"] == "United States"
].copy()

state_counts = (
    us_breweries["state"]
    .value_counts()
    .reset_index()
)

state_counts.columns = ["state", "count"]

state_counts.head(10)

,state,count
0,California,159
1,Washington,85
2,Colorado,70
3,Texas,69
4,Michigan,69
5,New York,60
6,Florida,55
7,Oregon,54
8,Virginia,53
9,North Carolina,50


In [12]:
top_states = state_counts.head(10)

fig = px.bar(
    top_states,
    x="state",
    y="count",
    title="Top 10 U.S. States by Number of Breweries",
    labels={
        "state": "State",
        "count": "Number of Breweries"
    },
    text="count"
)

fig.update_traces(textposition="outside")

fig.show()

### Interpretation

The distribution of breweries across states is uneven rather than uniform. California contains 159 breweries, almost twice as many as Washington, the second-highest state with 85 breweries. Colorado (70), Texas (69), and Michigan (69) also have relatively large numbers of breweries, while the remaining states contain substantially fewer.

These results indicate that brewery activity is concentrated in a relatively small number of states instead of being evenly distributed across the United States. This concentration suggests that regional factors, such as population, tourism, local culture, or state regulations, may influence where breweries are established.

## Finding 3: Brewery Locations Form Regional Clusters

State-level totals show that breweries are concentrated in particular states, but they do not reveal where breweries are located within the country. Mapping brewery locations provides a clearer picture of regional clustering and highlights areas where breweries are especially common.

In [13]:
fig = px.scatter_geo(
    map_df,
    lat="latitude",
    lon="longitude",
    hover_name="name",
    color="brewery_type",
    scope="usa",
    title="Geographic Distribution of Breweries in the United States"
)

fig.show()

### Interpretation

The geographic distribution shows that breweries are not spread evenly across the United States. Instead, breweries form clear regional clusters, particularly along the West Coast, in the Northeast, and around major metropolitan areas.

These clusters support the previous state-level analysis by showing that brewery activity is concentrated both across and within states. Together, the findings suggest that brewery location is influenced by regional demand and population patterns rather than being evenly distributed nationwide.

# Conclusion

This analysis explored how brewery characteristics vary across the United States using data collected from the Open Brewery DB API.

Three main findings emerged. First, brewery types are highly concentrated, with micro breweries representing the largest category by a substantial margin. Second, breweries are not evenly distributed across states, with California containing the highest number of breweries and several western and northeastern states also showing relatively high concentrations. Finally, mapping brewery locations revealed clear regional clusters that reinforce the state-level patterns.

Together, these findings indicate that the U.S. brewing industry is characterised by both business-type concentration and geographic clustering. These results provide evidence that brewery characteristics vary considerably across the country and establish the key insights that will be presented on the project's public website.